# 🏆 NegotiateAI — GRPO Training Notebook
### OpenEnv Hackathon Finale | April 2026

Train a Llama 3.2 3B model to negotiate enterprise procurement contracts using GRPO.

**What this notebook does:**
1. Connects to your live NegotiateAI environment on HuggingFace Spaces
2. Generates training episodes (agent takes actions, gets rewards)
3. Fine-tunes the model using GRPO via Unsloth + HF TRL
4. Shows reward curves proving training improvement
5. Runs before/after comparison for your pitch demo

**Runtime required:** GPU (T4 or better) — set in Runtime → Change runtime type

---


## Cell 1 — Install Dependencies

In [ ]:
# Install Unsloth (fast LLM fine-tuning) + TRL (GRPO) + dependencies
# This takes ~3-4 minutes on first run

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install trl>=0.12.0 requests numpy matplotlib -q

print("✅ Installation complete")


## Cell 2 — Configuration

⚠️ **Set your HuggingFace Space URL and OpenAI API key here before running.**

In [ ]:
import os

# ── SET THESE ──────────────────────────────────────────────────────────────────
ENV_URL      = "https://prasanthdj8-negotiateai-openenv.hf.space"   # your HF Space URL
OPENAI_KEY   = ""   # needed for supplier LLMs (set in HF Space secrets too)
HF_TOKEN     = ""   # your HuggingFace token for saving model

# ── TRAINING CONFIG ────────────────────────────────────────────────────────────
MODEL_NAME          = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH      = 1024
LOAD_IN_4BIT        = True          # fits on free T4 GPU

LEARNING_RATE       = 5e-6
NUM_EPOCHS          = 3
BATCH_SIZE          = 2             # small batch for T4 memory
NUM_GENERATIONS     = 4             # candidate actions per GRPO step
MAX_COMPLETION_LEN  = 300
LOGGING_STEPS       = 10

# Episode generation
EASY_EPISODES       = 200           # start with easy task
MEDIUM_EPISODES     = 100           # then medium
MAX_STEPS_PER_EP    = 30            # max steps per episode

SAVE_DIR            = "/content/negotiateai-grpo"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ Config ready")
print(f"   ENV_URL:    {ENV_URL}")
print(f"   Model:      {MODEL_NAME}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Generations:{NUM_GENERATIONS}")


## Cell 3 — Verify Environment Is Live

Run this to confirm your HF Space is responding before training.

In [ ]:
import requests, json

def env_health():
    try:
        r = requests.get(f"{ENV_URL}/health", timeout=15)
        return r.status_code == 200 and r.json().get("status") == "healthy"
    except Exception as e:
        print(f"  Error: {e}")
        return False

def env_reset(task_id="easy_negotiation", seed=None):
    r = requests.post(f"{ENV_URL}/reset",
        json={"task_id": task_id, "seed": seed}, timeout=30)
    r.raise_for_status()
    return r.json()

def env_step(action):
    r = requests.post(f"{ENV_URL}/step",
        json={"action": action}, timeout=30)
    r.raise_for_status()
    return r.json()

def env_tasks():
    r = requests.get(f"{ENV_URL}/tasks", timeout=15)
    return r.json()

# ── Health check ───────────────────────────────────────────────────────────────
print("Checking environment health...")
if env_health():
    print("✅ Environment is healthy!")
else:
    print("❌ Environment not reachable. Check your HF Space URL and ensure it's awake.")
    print("   Visit your HF Space URL in a browser first to wake it up.")

# ── Show available tasks ───────────────────────────────────────────────────────
tasks_data = env_tasks()
print(f"\n📋 Available tasks ({tasks_data['count']}):")
for t in tasks_data["tasks"]:
    print(f"   {t['task_id']:30s} baseline={t['baseline_score']} → target={t['target_score']}")

# ── Quick reset test ───────────────────────────────────────────────────────────
print("\nRunning quick reset test...")
reset_data = env_reset("easy_negotiation", seed=42)
obs = reset_data["observation"]
print(f"✅ Reset OK: week={obs['week']}/{obs['total_weeks']} "
      f"suppliers={len(obs['suppliers'])} "
      f"budget=${obs['budget_remaining']:,.0f}")


## Cell 4 — Prompt Templates

The system prompt tells the LLM what it is and what JSON format to output.
The user prompt converts each observation into a readable situation summary.

In [ ]:
SYSTEM_PROMPT = """You are an expert AI procurement agent for a tech company.
Your goal: source software, hardware, and services at the best cost,
within budget, before deadlines — while outperforming a rival buyer
and detecting deceptive suppliers.

STRATEGY GUIDE:
1. Always negotiate price DOWN from list — open below list price
2. Raise a PR before awarding contracts (compliance requirement)
3. Watch rival_pressure > 0.6 — hedge orders if this happens
4. Reject suppliers with community_rating < 0.65 or known issues
5. Escalate PRs when deadline ≤ 2 weeks away
6. Hedge critical items across 2 suppliers when rival pressure is high
7. Read supplier messages carefully — deceptive suppliers deflect on reliability

Respond with ONLY valid JSON. No markdown. No preamble:
{
  "action_type": "negotiate|award_contract|reject|raise_pr|escalate|hedge|defer|cancel_contract",
  "supplier_id": "<from observation>",
  "item_id": "<from requirements>",
  "message": "<natural language to supplier>",
  "proposed_price": <float or null>,
  "proposed_quantity": <int or null>,
  "proposed_lead_time": <int or null>,
  "hedge_supplier_id": "<second supplier for hedge or null>",
  "hedge_quantity": <int or null>,
  "notes": "<your reasoning>"
}""".strip()


def obs_to_prompt(obs):
    """Convert observation dict to LLM-readable prompt."""
    suppliers = obs.get("suppliers", [])
    requirements = obs.get("requirements", [])
    contracts = obs.get("contracts", [])
    signals = obs.get("market_signals", [])[-3:]
    disruptions = obs.get("disruptions", [])
    rival_activity = obs.get("rival_activity", {})

    sup_lines = []
    for s in suppliers:
        status = "ACTIVE" if s["is_active"] else "OFFLINE"
        flag = " ⚠️RIVAL" if s.get("rival_pressure", 0) > 0.6 else ""
        sup_lines.append(
            f"  {s['supplier_id']} | {s['name']} ({s['category']}) | "
            f"${s['base_price']:.0f}/unit | lead={s['lead_time_days']}d | "
            f"rel={s['reliability_score']:.2f} | cap={s['capacity_available']} | "
            f"{status}{flag}"
        )

    req_lines = []
    for r in requirements:
        done = "DONE" if r["fulfilled"] else "PENDING"
        crit = "[CRITICAL]" if r["is_critical"] else ""
        wleft = r["deadline_week"] - obs.get("week", 1)
        req_lines.append(
            f"  {r['item_id']} | {r['name']} | qty={r['quantity']} | "
            f"deadline=wk{r['deadline_week']}({wleft}w left) | "
            f"budget=${r['budget_ceiling']:,.0f} | {done} {crit}"
        )

    sig_lines = [
        f"  [{s['signal_type']}] {s['description'][:70]}"
        for s in signals
    ]

    dis_lines = [f"  ⚡ {d['description']}" for d in disruptions]

    rival_high = [
        f"{sid}(p={p:.2f})"
        for sid, p in rival_activity.items() if p > 0.4
    ]

    return f"""WEEK {obs.get('week','?')}/{obs.get('total_weeks','?')} | BUDGET ${obs.get('budget_remaining',0):,.0f} / ${obs.get('budget_total',0):,.0f}

REQUIREMENTS:
{chr(10).join(req_lines) or '  None'}

SUPPLIERS:
{chr(10).join(sup_lines) or '  None'}

CONTRACTS ACTIVE: {len([c for c in contracts if c.get('status') == 'active'])}

SIGNALS:
{chr(10).join(sig_lines) or '  None'}

DISRUPTIONS:
{chr(10).join(dis_lines) or '  None'}

RIVAL HIGH PRESSURE: {', '.join(rival_high) or 'Low'}
RIVAL CONTRACTS WON: {obs.get('rival_contracts_won', 0)}

Respond with JSON action only.""".strip()


print("✅ Prompts ready")
print(f"   System prompt: {len(SYSTEM_PROMPT)} chars")

# Quick prompt test
reset_data = env_reset("easy_negotiation", seed=1)
test_prompt = obs_to_prompt(reset_data["observation"])
print(f"   Sample prompt: {len(test_prompt)} chars")
print("   Preview:")
print("   " + "\n   ".join(test_prompt.split("\n")[:5]))


## Cell 5 — Generate Training Episodes

Run the environment with random actions to collect (prompt, reward) pairs for GRPO.
This exploration phase is important — GRPO needs diverse trajectories to learn from.

In [ ]:
import random
import numpy as np

ACTION_TYPES = [
    "negotiate", "raise_pr", "award_contract",
    "defer", "reject", "hedge", "escalate"
]

def random_action(obs):
    """Sample a random valid action for exploration."""
    suppliers = [s for s in obs.get("suppliers", []) if s["is_active"]]
    requirements = [r for r in obs.get("requirements", []) if not r["fulfilled"]]

    if not suppliers:
        suppliers = obs.get("suppliers", [])
    if not requirements:
        requirements = obs.get("requirements", [])
    if not suppliers or not requirements:
        return None

    sup = random.choice(suppliers)
    req = random.choice(requirements)
    atype = random.choice(ACTION_TYPES)

    action = {
        "action_type":       atype,
        "supplier_id":       sup["supplier_id"],
        "item_id":           req["item_id"],
        "message":           f"Exploring {atype} with {sup['name']} for {req['name']}",
        "proposed_price":    round(sup["base_price"] * random.uniform(0.80, 1.05), 2),
        "proposed_quantity": req["quantity"],
        "notes":             "Random exploration episode",
    }

    # Add hedge fields if needed
    if atype == "hedge" and len(suppliers) >= 2:
        other = random.choice([s for s in suppliers if s["supplier_id"] != sup["supplier_id"]])
        action["hedge_supplier_id"] = other["supplier_id"]
        action["hedge_quantity"]    = max(1, req["quantity"] // 2)

    return action


def generate_episodes(task_id="easy_negotiation", n_episodes=200,
                       max_steps=30, seed=42):
    """
    Generate training episodes via random exploration.
    Returns list of {prompt, reward} dicts for GRPO dataset.
    """
    random.seed(seed)
    dataset = []
    episode_scores = []

    print(f"Generating {n_episodes} episodes for {task_id}...")
    print(f"  Max steps per episode: {max_steps}")

    for ep in range(n_episodes):
        try:
            reset_data = env_reset(task_id, seed=seed + ep)
            obs = reset_data["observation"]
            episode_rewards = []

            for step_i in range(max_steps):
                if obs.get("done", False):
                    break

                action = random_action(obs)
                if action is None:
                    break

                # Build training sample BEFORE stepping
                prompt = obs_to_prompt(obs)
                messages = [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": prompt},
                    {"role": "assistant", "content": json.dumps(action)},
                ]

                # Step environment
                step_data = env_step(action)
                reward = step_data["reward"]["total"]
                obs = step_data["observation"]
                episode_rewards.append(reward)

                # Add to dataset
                dataset.append({
                    "prompt": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": prompt},
                    ],
                    "reward": reward,
                    "task_id": task_id,
                    "episode": ep,
                    "step":    step_i,
                })

            if episode_rewards:
                episode_scores.append(np.mean(episode_rewards))

            if (ep + 1) % 20 == 0:
                avg = np.mean(episode_scores[-20:]) if episode_scores else 0
                print(f"  Episode {ep+1:4d}/{n_episodes} | "
                      f"avg_reward={avg:.4f} | "
                      f"dataset_size={len(dataset)}")

        except Exception as e:
            print(f"  Episode {ep}: error — {e}")
            continue

    print(f"\n✅ Generated {len(dataset)} training samples from {n_episodes} episodes")
    print(f"   Reward stats: mean={np.mean(episode_scores):.4f} "
          f"min={np.min(episode_scores):.4f} "
          f"max={np.max(episode_scores):.4f}")
    return dataset, episode_scores


# ── Generate easy episodes ─────────────────────────────────────────────────────
easy_dataset, easy_scores = generate_episodes(
    task_id="easy_negotiation",
    n_episodes=EASY_EPISODES,
    max_steps=MAX_STEPS_PER_EP,
    seed=42,
)
print(f"\n📊 Dataset ready: {len(easy_dataset)} samples")


## Cell 6 — Load Model with Unsloth

Loads Llama 3.2 3B in 4-bit quantisation — fits on a free T4 GPU (15GB VRAM).

In [ ]:
from unsloth import FastLanguageModel
import torch

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (no GPU detected!)'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

print(f"\nLoading {MODEL_NAME}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,   # auto-detect
)

# Add LoRA adapters — only train a small fraction of weights
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                              # LoRA rank
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("\n✅ Model loaded")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"   Total params:     {sum(p.numel() for p in model.parameters()):,}")

# Verify tokenizer
test_tokens = tokenizer("Test procurement action", return_tensors="pt")
print(f"   Tokenizer OK: {test_tokens['input_ids'].shape}")


## Cell 7 — GRPO Reward Function

This is the core of training. GRPO generates multiple candidate actions,
calls the environment for each, and updates the model toward higher-scoring ones.

In [ ]:
import math

# Track rewards for plotting
grpo_step_rewards = []
grpo_rolling_avgs = []


def procurement_reward_fn(completions, prompts=None, **kwargs):
    """
    Reward function called by GRPOTrainer each training step.

    For each candidate completion (LLM action):
    1. Parse JSON action
    2. Reset environment and step with the action
    3. Return the reward score
    4. GRPO updates model toward higher-scoring completions
    """
    rewards = []

    for completion in completions:
        # Extract text from completion
        if isinstance(completion, list):
            text = completion[0].get("content", "") if completion else ""
        else:
            text = str(completion)

        # Strip markdown fences
        text = text.strip()
        if text.startswith("```"):
            parts = text.split("```")
            text = parts[1] if len(parts) > 1 else text
            if text.startswith("json"):
                text = text[4:].strip()

        try:
            action = json.loads(text)

            # Quick validation — must have required fields
            if not all(k in action for k in ["action_type", "supplier_id", "item_id", "message"]):
                rewards.append(1e-4)
                continue

            # Reset env and step with this action
            reset_data = env_reset("easy_negotiation", seed=random.randint(0, 999))
            obs = reset_data["observation"]

            # Validate supplier and item exist
            valid_sups  = {s["supplier_id"] for s in obs["suppliers"]}
            valid_items = {r["item_id"]      for r in obs["requirements"]}

            if action["supplier_id"] not in valid_sups:
                action["supplier_id"] = obs["suppliers"][0]["supplier_id"]
            if action["item_id"] not in valid_items:
                action["item_id"] = obs["requirements"][0]["item_id"]

            step_data = env_step(action)
            reward = float(np.clip(
                step_data["reward"]["total"], 1e-4, 1 - 1e-4
            ))
            rewards.append(reward)

        except json.JSONDecodeError:
            rewards.append(1e-4)       # invalid JSON = minimum reward
        except Exception as e:
            rewards.append(1e-4)

    # Track for reward curve
    if rewards:
        avg = np.mean(rewards)
        grpo_step_rewards.extend(rewards)
        grpo_rolling_avgs.append(avg)

    return rewards


# Quick reward function test
print("Testing reward function...")
test_completions = [
    [{"content": json.dumps({
        "action_type": "negotiate",
        "supplier_id": "sup_001",
        "item_id": "item_001",
        "message": "We need 50 licenses at $950 each",
        "proposed_price": 950.0,
        "proposed_quantity": 50,
        "notes": "Test"
    })}],
    [{"content": "invalid json here"}],
]
test_rewards = procurement_reward_fn(test_completions)
print(f"✅ Reward function works")
print(f"   Valid action reward:   {test_rewards[0]:.4f}")
print(f"   Invalid JSON reward:   {test_rewards[1]:.4f}")


## Cell 8 — GRPO Training

⏱️ **Estimated time: 1.5–2.5 hours on T4 GPU**

Watch the reward curve climb in the logs. Every 10 steps you'll see the average reward.
Target: reward rising from ~0.10 to ~0.40+ by end of training.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

# Convert episode dataset to HF Dataset format
hf_dataset = Dataset.from_list([
    {"prompt": ep["prompt"]}
    for ep in easy_dataset
])

print(f"Training dataset: {len(hf_dataset)} samples")
print(f"First sample prompt preview:")
print(hf_dataset[0]["prompt"][1]["content"][:200] + "...")

# ── GRPO Config ────────────────────────────────────────────────────────────────
grpo_config = GRPOConfig(
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LEN,
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION_LEN,
    output_dir=SAVE_DIR,
    logging_steps=LOGGING_STEPS,
    save_steps=50,
    warmup_ratio=0.05,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

# ── Trainer ────────────────────────────────────────────────────────────────────
trainer = GRPOTrainer(
    model=model,
    reward_funcs=procurement_reward_fn,
    args=grpo_config,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
)

print("\n🚀 Starting GRPO training...")
print(f"   Model:       {MODEL_NAME}")
print(f"   Dataset:     {len(hf_dataset)} samples")
print(f"   Epochs:      {NUM_EPOCHS}")
print(f"   Batch size:  {BATCH_SIZE}")
print(f"   Generations: {NUM_GENERATIONS}")
print(f"   LR:          {LEARNING_RATE}")
print()

trainer.train()

print("\n✅ Training complete!")
print(f"   Steps completed: {trainer.state.global_step}")

# Save model
model.save_pretrained(f"{SAVE_DIR}/final")
tokenizer.save_pretrained(f"{SAVE_DIR}/final")
print(f"   Model saved to: {SAVE_DIR}/final")


## Cell 9 — Plot Reward Curve

This is the chart you show judges. Clear upward trend = training is working.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Get reward log from trainer
log_history = trainer.state.log_history
steps  = [l["step"]  for l in log_history if "loss" in l]
losses = [l["loss"]  for l in log_history if "loss" in l]

# Also get our manual reward tracking
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("NegotiateAI — GRPO Training Results", fontsize=14, fontweight="bold")

# ── Plot 1: Step rewards ───────────────────────────────────────────────────────
ax1 = axes[0]
if grpo_step_rewards:
    # Rolling average
    window = 20
    rolling = []
    for i in range(len(grpo_step_rewards)):
        start = max(0, i - window)
        rolling.append(np.mean(grpo_step_rewards[start:i+1]))

    ax1.scatter(range(len(grpo_step_rewards)), grpo_step_rewards,
                alpha=0.2, s=8, color="#94a3b8", label="Step reward")
    ax1.plot(rolling, color="#3b82f6", linewidth=2, label=f"Rolling avg (n={window})")

    # Baseline reference line
    ax1.axhline(y=0.15, color="#ef4444", linestyle="--",
                linewidth=1.5, label="Random baseline (0.15)", alpha=0.7)
    ax1.axhline(y=0.60, color="#22c55e", linestyle="--",
                linewidth=1.5, label="Target (0.60)", alpha=0.7)

    ax1.set_xlabel("Training Step")
    ax1.set_ylabel("Reward")
    ax1.set_title("Step-Level Reward Progression")
    ax1.legend(fontsize=9)
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.3)

# ── Plot 2: Training loss ──────────────────────────────────────────────────────
ax2 = axes[1]
if steps and losses:
    ax2.plot(steps, losses, color="#8b5cf6", linewidth=2)
    ax2.set_xlabel("Training Step")
    ax2.set_ylabel("Loss")
    ax2.set_title("Training Loss")
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "Loss data\nnot available",
             ha="center", va="center", transform=ax2.transAxes, fontsize=12)
    ax2.set_title("Training Loss")

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Print summary stats ────────────────────────────────────────────────────────
if grpo_step_rewards:
    first_20  = np.mean(grpo_step_rewards[:20])
    last_20   = np.mean(grpo_step_rewards[-20:])
    improvement = ((last_20 - first_20) / max(first_20, 1e-6)) * 100
    print(f"\n📊 Training Summary:")
    print(f"   First 20 steps avg reward:  {first_20:.4f}")
    print(f"   Last  20 steps avg reward:  {last_20:.4f}")
    print(f"   Improvement:                +{improvement:.1f}%")
    print(f"   Chart saved: {SAVE_DIR}/reward_curve.png")


## Cell 10 — Before vs After Demo

This is what you show judges in the pitch. Side-by-side comparison of untrained vs trained agent decisions.

In [ ]:
from unsloth import FastLanguageModel

def run_inference(model, tokenizer, obs, max_new_tokens=300):
    """Run one inference step — returns action dict."""
    prompt = obs_to_prompt(obs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        FastLanguageModel.for_inference(model)
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()

    try:
        return json.loads(generated), generated
    except:
        return None, generated


def run_demo_episode(model, tokenizer, task_id="easy_negotiation",
                     seed=999, max_steps=8, label="Agent"):
    """Run a short episode and return decisions + score."""
    reset_data = env_reset(task_id, seed=seed)
    obs = reset_data["observation"]
    actions_taken = []
    total_reward = 0

    print(f"\n{'─'*60}")
    print(f"{label} — {task_id} (seed={seed})")
    print(f"Week 1/{obs['total_weeks']} | Budget: ${obs['budget_remaining']:,.0f}")
    print(f"{'─'*60}")

    for step_i in range(max_steps):
        if obs.get("done", False):
            break

        action, raw = run_inference(model, tokenizer, obs)

        if action is None:
            print(f"  Step {step_i+1}: ❌ Invalid JSON output")
            action = {
                "action_type":  "defer",
                "supplier_id":  obs["suppliers"][0]["supplier_id"],
                "item_id":      obs["requirements"][0]["item_id"],
                "message":      "Deferring",
            }

        step_data = env_step(action)
        reward = step_data["reward"]["total"]
        total_reward += reward
        obs = step_data["observation"]

        print(f"  Step {step_i+1}: {action['action_type']:20s} | "
              f"reward={reward:.4f} | "
              f"{step_data['reward'].get('explanation', '')[:45]}")

        if action.get("notes"):
            print(f"          Notes: {action['notes'][:60]}")

        actions_taken.append({
            "step": step_i + 1, "action": action["action_type"],
            "reward": reward, "notes": action.get("notes", ""),
        })

    avg_reward = total_reward / max(len(actions_taken), 1)
    print(f"\n  Avg reward: {avg_reward:.4f}")
    return actions_taken, avg_reward


# ── Run before/after on same scenario ─────────────────────────────────────────
DEMO_SEED  = 777
DEMO_STEPS = 6

print("=" * 60)
print("BEFORE vs AFTER TRAINING COMPARISON")
print("=" * 60)

# Load base model (before training) for comparison
print("\nLoading base model (untrained)...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
base_actions, base_score = run_demo_episode(
    base_model, base_tokenizer,
    seed=DEMO_SEED, max_steps=DEMO_STEPS,
    label="❌ UNTRAINED AGENT"
)
del base_model  # free VRAM

# Run trained model
print()
trained_actions, trained_score = run_demo_episode(
    model, tokenizer,
    seed=DEMO_SEED, max_steps=DEMO_STEPS,
    label="✅ TRAINED AGENT"
)

# ── Summary ────────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("RESULTS")
print("=" * 60)
improvement = ((trained_score - base_score) / max(base_score, 1e-6)) * 100
print(f"  Untrained avg reward: {base_score:.4f}")
print(f"  Trained   avg reward: {trained_score:.4f}")
print(f"  Improvement:          +{improvement:.1f}%")
print("=" * 60)
print()
print("🎯 This is your pitch demo. Show this output to judges.")


## Cell 11 — Train on Medium Task (Optional)

Only run this if you have time after easy task training looks good.
The medium task includes deceptive suppliers and a rival buyer.

In [ ]:
# Generate medium episodes
print("Generating medium_adversarial episodes...")
medium_dataset, medium_scores = generate_episodes(
    task_id="medium_adversarial",
    n_episodes=MEDIUM_EPISODES,
    max_steps=MAX_STEPS_PER_EP,
    seed=100,
)

# Fine-tune on medium task (continuing from easy-trained model)
medium_hf_dataset = Dataset.from_list([
    {"prompt": ep["prompt"]} for ep in medium_dataset
])

medium_config = GRPOConfig(
    learning_rate=LEARNING_RATE / 2,    # lower LR for continued training
    num_train_epochs=2,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LEN,
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION_LEN,
    output_dir=f"{SAVE_DIR}/medium",
    logging_steps=LOGGING_STEPS,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

medium_trainer = GRPOTrainer(
    model=model,
    reward_funcs=procurement_reward_fn,
    args=medium_config,
    train_dataset=medium_hf_dataset,
    processing_class=tokenizer,
)

print("\n🚀 Training on medium task...")
medium_trainer.train()
model.save_pretrained(f"{SAVE_DIR}/medium_final")
print("✅ Medium training done")


## Cell 12 — Push Model to HuggingFace Hub (Optional)

Save the trained model to your HuggingFace profile for the blog post.

In [ ]:
# Optional: push trained model to HF Hub
# Set HF_TOKEN at the top of this notebook

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

    model.push_to_hub(
        "prasanthdj8/negotiateai-procurement-agent",
        token=HF_TOKEN,
    )
    tokenizer.push_to_hub(
        "prasanthdj8/negotiateai-procurement-agent",
        token=HF_TOKEN,
    )
    print("✅ Model pushed to HuggingFace Hub")
    print("   https://huggingface.co/prasanthdj8/negotiateai-procurement-agent")
else:
    print("⚠️  HF_TOKEN not set — skipping Hub push")
    print("   Set HF_TOKEN in Cell 2 and re-run if needed")


## Cell 13 — Pitch Day Checklist

Run this to confirm everything is ready before you walk on stage.


In [ ]:
import os

print("=" * 60)
print("PITCH DAY CHECKLIST")
print("=" * 60)

checks = []

# 1. Environment health
healthy = env_health()
checks.append(("HF Space live & healthy",    healthy))

# 2. Model saved
model_saved = os.path.exists(f"{SAVE_DIR}/final/adapter_config.json")
checks.append(("Trained model saved locally", model_saved))

# 3. Reward curve saved
curve_saved = os.path.exists(f"{SAVE_DIR}/reward_curve.png")
checks.append(("Reward curve chart saved",    curve_saved))

# 4. Training happened
trained = len(grpo_step_rewards) > 0
checks.append(("GRPO training completed",     trained))

# 5. Improvement shown
if grpo_step_rewards and len(grpo_step_rewards) >= 40:
    first = np.mean(grpo_step_rewards[:20])
    last  = np.mean(grpo_step_rewards[-20:])
    improved = last > first
    checks.append((f"Reward improved ({first:.3f} → {last:.3f})", improved))

print()
for label, passed in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {label}")

print()
all_passed = all(p for _, p in checks)
if all_passed:
    print("🎉 All checks passed — you're ready to pitch!")
else:
    failed = [l for l, p in checks if not p]
    print(f"⚠️  {len(failed)} check(s) need attention before your pitch.")

print()
print("Pitch story:")
if grpo_step_rewards and len(grpo_step_rewards) >= 40:
    first = np.mean(grpo_step_rewards[:20])
    last  = np.mean(grpo_step_rewards[-20:])
    pct = ((last - first) / max(first, 1e-6)) * 100
    print(f'  "Our trained agent improved from {first:.2f} to {last:.2f} '
          f'— a {pct:.0f}% improvement over the random baseline."')
